In [ ]:
import shap
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
from lime.lime_text import LimeTextExplainer

/Users/mikiyasegaye/MK_Lab/10 Academy/Amharic-E-commerce-Data-Extractor/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_path = "amharic-ner-model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)
ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

Device set to use mps:0


In [ ]:
class NERPredictWrapper:
    def __init__(self, pipeline):
        self.pipeline = pipeline
        self.labels = self._get_all_labels()

    def _get_all_labels(self):
        labels = set()
        for example in sample_texts:
            for item in self.pipeline(example):
                labels.add(item["entity_group"])
        return sorted(list(labels))

    def predict_proba(self, texts):
        result = []
        for text in texts:
            entities = self.pipeline(text)
            label_scores = {label: 0.0 for label in self.labels}
            for ent in entities:
                label_scores[ent["entity_group"]] += ent["score"]
            total = sum(label_scores.values())
            normalized = [label_scores[label] / total if total > 0 else 0.0 for label in self.labels]
            result.append(normalized)
        return np.array(result)

    def class_names(self):
        return self.labels

In [14]:
from lime.lime_text import LimeTextExplainer
from transformers import pipeline
import numpy as np

ner_pipeline = pipeline("ner", model="amharic-ner-model", tokenizer="amharic-ner-model", aggregation_strategy="simple")

class_names = ["no-entity", "has-entity"]

def predict_fn(texts):
    outputs = []
    for text in texts:
        result = ner_pipeline(text)
        prob_has_entity = 1.0 if result else 0.0
        outputs.append([1 - prob_has_entity, prob_has_entity])
    return np.array(outputs)

explainer = LimeTextExplainer(class_names=class_names, bow=False, char_level=True)

sample_text = "አዲስ ልብስ በቦሌ ላይ በ 250 ብር ሽያጭ ይካሄዳል"
exp = explainer.explain_instance(sample_text, predict_fn, num_features=10)

for feature, weight in exp.as_list():
    print(f"{feature}: {weight:.4f}")

Device set to use mps:0


አ: 0.0000
ዲ: 0.0000
ስ: 0.0000
 : 0.0000
ል: 0.0000
ብ: 0.0000
ስ: 0.0000
 : 0.0000
በ: 0.0000
ቦ: 0.0000


/Users/mikiyasegaye/MK_Lab/10 Academy/Amharic-E-commerce-Data-Extractor/venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/mikiyasegaye/MK_Lab/10 Academy/Amharic-E-commerce-Data-Extractor/venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/mikiyasegaye/MK_Lab/10 Academy/Amharic-E-commerce-Data-Extractor/venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/mikiyasegaye/MK_Lab/10 Academy/Amharic-E-commerce-Data-Extractor/venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/mikiyasegaye/MK_Lab/10 Academy/Amharic-E-commerce-Data-Extractor/venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ 

In [9]:
from shap import Explanation

def shap_ner_explanation(text, pipeline):
    explainer = shap.Explainer(pipeline, masker=shap.maskers.Text(tokenizer))
    shap_values = explainer([text])
    shap.plots.text(shap_values[0])

for text in sample_texts:
    shap_ner_explanation(text, ner_pipeline)


🚨 `input_features` is part of SeamlessM4TForSpeechToText.forward's signature, but not documented. Make sure to add it to the docstring of the function in /Users/mikiyasegaye/MK_Lab/10 Academy/Amharic-E-commerce-Data-Extractor/venv/lib/python3.13/site-packages/transformers/models/seamless_m4t/modeling_seamless_m4t.py.
🚨 `input_features` is part of SeamlessM4TForSpeechToSpeech.forward's signature, but not documented. Make sure to add it to the docstring of the function in /Users/mikiyasegaye/MK_Lab/10 Academy/Amharic-E-commerce-Data-Extractor/venv/lib/python3.13/site-packages/transformers/models/seamless_m4t/modeling_seamless_m4t.py.
🚨 `input_features` is part of SeamlessM4TModel.forward's signature, but not documented. Make sure to add it to the docstring of the function in /Users/mikiyasegaye/MK_Lab/10 Academy/Amharic-E-commerce-Data-Extractor/venv/lib/python3.13/site-packages/transformers/models/seamless_m4t/modeling_seamless_m4t.py.
🚨 `input_features` is part of SeamlessM4Tv2ForSpeec